# 07 - Rolling-horizon EMS and nominal comparison B0 / B1 / B2 / DFM (workflow steps 8-10)

**Questions:** (i) is the EMS implementation correct (executed action inside
$\mathcal F^{safe}_k$, no simultaneous charge/discharge, move blocking faithful to the full
resolution)? (ii) Does the dynamic envelope differ from the rated capability over time, and which
mechanism limits it (Figure 4 -- the key figure)? (iii) How do the four strategies compare on the
same nominal 24-h trajectory (Table 3, Figures 5-6)?

Outputs: `output/results/log_nominal_<strategy>.csv`, `output/results/log_trial0_DFM.csv`,
`output/results/results_nominal_summary.csv`, `output/results/results_ems_validation.csv`,
`output/experiment_figures/Fig_flexibility_evolution.pdf`,
`output/experiment_figures/Fig_energy_performance.pdf`, `output/experiment_figures/Fig_network_security.pdf`.

In [ ]:
import sys, os, time
from dataclasses import replace
sys.path.insert(0, os.path.abspath(".."))
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from dfm_acdc.config import RESULTS_DIR, FIG_DIR, MCParams
from dfm_acdc.simulate import Study, summarize
from dfm_acdc.ems import strategies
from dfm_acdc.uncertainty import sample_scenarios
from dfm_acdc import plotstyle

plotstyle.apply_style()
S = Study()
S.load_or_precompute(os.path.join(RESULTS_DIR, "precompute_network_forecast.csv"))
P_res = S.reserve()
ST = strategies()
b, dt = S.sp.b, S.sp.ems.dt_h
tests = []
def check(name, value, ok, tol=""):
    tests.append({"test": name, "value": value, "tolerance": tol, "pass": bool(ok)})
    print(f"[{'PASS' if ok else 'FAIL'}] {name}: {value}")

## Nominal runs (identical trajectory, BESS rating and initial state for all strategies)

In [ ]:
logs, summ = {}, []
for name in ("B0", "B1", "B2", "DFM"):
    t0 = time.perf_counter()
    logs[name] = S.run(ST[name], P_res=P_res)
    s = summarize(logs[name], S); s["wall_s"] = time.perf_counter() - t0
    summ.append(s)
    logs[name].to_csv(os.path.join(RESULTS_DIR, f"log_nominal_{name}.csv"), index=False)
nom = pd.DataFrame(summ).set_index("strategy")
nom.to_csv(os.path.join(RESULTS_DIR, "results_nominal_summary.csv"))
cols = ["E_grid_kWh", "E_grid_corr_kWh", "cost_corr_EUR", "E_curt_kWh", "eta_RES", "E_loss_kWh", "dSOH_rainflow",
        "N_EFC", "n_viol_intervals", "max_exceedance", "soc_final", "t_solve_mean_ms", "t_solve_p95_ms"]
print(nom[cols].T.to_string())

## EMS validation

In [ ]:
for name in ("B1", "B2", "DFM"):
    L = logs[name]
    check(f"{name}: no simultaneous charge and discharge", float((L.P_dis * L.P_ch).max()), (L.P_dis * L.P_ch).max() == 0.0)
    check(f"{name}: SOC within [SOC_min, SOC_max]", f"[{L.soc_next.min():.3f}, {L.soc_next.max():.3f}]",
          L.soc_next.min() >= b.soc_min - 1e-9 and L.soc_next.max() <= b.soc_max + 1e-9)
    check(f"{name}: all MILPs solved to optimality (no time-limit hit)", f"{int((L.ems_status != 0).sum())} non-optimal",
          (L.ems_status == 0).all())
L = logs["DFM"]
inside = ((L.P_sched <= L.st_P_safe_dis + 1e-6) & (L.P_sched >= -L.st_P_safe_ch - 1e-6)).all()
check("DFM: executed schedule inside F_safe at every interval", "96/96" if inside else "violated", inside)
phys = (np.minimum.reduce([L.st_P_en_dis, L.st_P_deg_dis, L.st_P_net_dis]) == L.st_P_flex_dis).all()
check("DFM: P_flex = min(P_en, P_deg, P_net) along the closed loop", "", phys)
for name in ("B1", "B2", "DFM"):
    r = nom.loc[name]
    check(f"{name}: online vs rainflow cycle stress on the actual dispatch",
          f"{r.F_cyc_online:.3e} / {r.F_cyc_rainflow:.3e} (ratio {r.F_cyc_online / r.F_cyc_rainflow:.3f})",
          0.8 < r.F_cyc_online / r.F_cyc_rainflow < 1.25, "+/-25 %")

### Move blocking vs. full 15-min resolution (DFM, nominal)

In [ ]:
S_full = Study(net_model=S.net); S_full.pre = S.pre
S_full.sp.ems = replace(S.sp.ems, n_fine_steps=96)
t0 = time.perf_counter()
L_full = S_full.run(ST["DFM"], P_res=P_res, record_envelope=False)
s_full = summarize(L_full, S_full)
S.sp.ems = replace(S.sp.ems)  # (unchanged default for the rest)
mb = pd.DataFrame({"move_blocking": nom.loc["DFM", cols], "full_resolution": pd.Series(s_full)[cols]})
print(mb.to_string())
rel = abs(s_full["cost_corr_EUR"] - nom.loc["DFM", "cost_corr_EUR"]) / s_full["cost_corr_EUR"]
check("move blocking: closed-loop cost within 0.1 % of the full-resolution MILP", f"{rel*100:.3f} %", rel < 1e-3, "0.1 %")
check("move blocking: solve time reduction", f"{s_full['t_solve_mean_ms']:.0f} -> {nom.loc['DFM','t_solve_mean_ms']:.0f} ms (mean)",
      nom.loc["DFM", "t_solve_mean_ms"] < s_full["t_solve_mean_ms"])

### One uncertain realization (Monte Carlo trial #0 -- the first sample, not selected)

In [ ]:
mc = MCParams()
sc = sample_scenarios(S.day, S.sp.unc, S.sp.ren.wt_rated_kw, mc.n_trials, mc.seed)
real0 = {"P_pv": sc.P_pv[0], "P_wt": sc.P_wt[0], "mult_ac": sc.mult_ac[0], "mult_dc": sc.mult_dc[0], "eps_net": sc.eps_net[0]}
L0 = S.run(ST["DFM"], real=real0, P_res=P_res)
L0.to_csv(os.path.join(RESULTS_DIR, "log_trial0_DFM.csv"), index=False)
inside0 = ((L0.P_B <= L0.st_P_flex_dis + 1e-6) & (L0.P_B >= -L0.st_P_flex_ch - 1e-6) |
           L0.rt_net_limited).all()
check("trial 0: executed power (schedule + correction) within battery-side physical limits", "", (
      (L0.P_B <= np.minimum(L0.st_P_en_dis, L0.st_P_deg_dis) + 1e-6) & (L0.P_B >= -np.minimum(L0.st_P_en_ch, L0.st_P_deg_ch) - 1e-6)).all())
print("active limiters (discharge side), nominal:", L.st_lim_dis.value_counts().to_dict())
print("active limiters (charge side),    nominal:", L.st_lim_ch.value_counts().to_dict())
print("active limiters (discharge side), trial 0:", L0.st_lim_dis.value_counts().to_dict())
print("active limiters (charge side),    trial 0:", L0.st_lim_ch.value_counts().to_dict())

In [ ]:
val = pd.DataFrame(tests)
val.to_csv(os.path.join(RESULTS_DIR, "results_ems_validation.csv"), index=False)
print(f"\n{val['pass'].sum()}/{len(val)} tests passed")
assert val["pass"].all()

## Figure 4 -- evolution of the dynamic flexibility envelope (key figure)

In [ ]:
def draw_envelope_column(axs, L, letters):
    t = L.t_h.values
    tt = np.r_[t, 24.0]
    P = b.P_rated_kw
    ax = axs[0]
    ax.plot(tt, np.r_[L.soc.values, L.soc_next.values[-1]], color=plotstyle.CAT["blue"])
    ax.axhline(b.soc_min, color=plotstyle.LIMIT_COLOR, lw=0.8, ls="--"); ax.axhline(b.soc_max, color=plotstyle.LIMIT_COLOR, lw=0.8, ls="--")
    ax.set_ylim(0, 1)
    ax = axs[1]
    ax.step(t, L.D_allow * 1e6, where="post", color=plotstyle.CAT["orange"])
    ax.set_ylim(0, L.D_max_cyc.max() * 1e6 * 1.15)
    ax.text(0.98, 0.88, rf"$\Delta SOH$ = {summarize(L, S)['dSOH_rainflow']*1e5:.2f}$\times10^{{-5}}$",
            transform=ax.transAxes, ha="right", va="top", fontsize=6.5, color=plotstyle.INK_SECONDARY)
    ax = axs[2]
    ax.axhline(P, color=plotstyle.INK_MUTED, lw=0.9, ls=":"); ax.axhline(-P, color=plotstyle.INK_MUTED, lw=0.9, ls=":")
    ax.fill_between(t, -L.st_P_flex_ch, L.st_P_flex_dis, step="post", color=plotstyle.CAT["violet"], alpha=0.16, lw=0)
    ax.fill_between(t, -L.st_P_safe_ch, L.st_P_safe_dis, step="post", color=plotstyle.CAT["aqua"], alpha=0.38, lw=0)
    ax.step(t, L.P_sched, where="post", color=plotstyle.INK_PRIMARY, lw=1.0)
    if (L.eps_net.abs() > 0).any():
        ax.step(t, L.P_B, where="post", color=plotstyle.CAT["orange"], lw=0.9)
    ax.set_ylim(-P - 20, P + 20)
    ax = axs[3]
    for row, col in ((1, "st_lim_dis"), (0, "st_lim_ch")):
        for ti, lab in zip(t, L[col]):
            ax.add_patch(plt.Rectangle((ti, row + 0.1), dt, 0.8, color=plotstyle.LIMITER_COLORS[lab], lw=0))
    ax.set_ylim(0, 2); ax.set_yticks([0.5, 1.5]); ax.set_yticklabels(["ch.", "dis."], fontsize=7); ax.grid(False)
    ax.set_xlim(0, 24); ax.set_xticks(range(0, 25, 4)); ax.set_xlabel("Time [h]")
    for a, lab in zip(axs, letters):
        a.text(0.01, 0.97, f"({lab})", transform=a.transAxes, fontsize=7.5, va="top", ha="left",
               bbox=dict(facecolor="white", edgecolor="none", pad=0.6, alpha=0.8))


FIG_WIDTH, FIG_HEIGHT = plotstyle.IEEE_PAGE_WIDTH_IN, 6.4
fig, axs = plt.subplots(4, 2, figsize=(FIG_WIDTH, FIG_HEIGHT), sharex=True,
                        gridspec_kw={"height_ratios": [0.8, 0.8, 1.7, 0.45], "hspace": 0.10, "wspace": 0.10})
draw_envelope_column(axs[:, 0], logs["DFM"], "abcd")
draw_envelope_column(axs[:, 1], L0, "efgh")
for r in range(4):
    axs[r, 1].sharey(axs[r, 0]); axs[r, 1].tick_params(labelleft=False)
axs[0, 0].set_ylabel("SOC"); axs[1, 0].set_ylabel(r"$D^{allow}_k$ [$10^{-6}$]")
axs[2, 0].set_ylabel("BESS power [kW]")
axs[0, 0].set_title("Nominal forecast (deterministic)", fontsize=8)
axs[0, 1].set_title("Uncertain realization (Monte Carlo trial #0)", fontsize=8)
from matplotlib.patches import Patch
from matplotlib.lines import Line2D
h1 = [Line2D([], [], color=plotstyle.INK_MUTED, ls=":", label=r"$\mathcal{F}^{rated}$"),
      Patch(facecolor=plotstyle.CAT["violet"], alpha=0.3, label=r"$\mathcal{F}^{physical}_k$"),
      Patch(facecolor=plotstyle.CAT["aqua"], alpha=0.5, label=r"$\mathcal{F}^{safe}_k$"),
      Line2D([], [], color=plotstyle.INK_PRIMARY, label=r"$P_{B,k}$ scheduled (EMS)"),
      Line2D([], [], color=plotstyle.CAT["orange"], label=r"$P_{B,k}$ executed (with correction)")]
fig.subplots_adjust(left=0.09, right=0.99, top=0.91, bottom=0.12)
fig.legend(handles=h1, loc="upper center", ncol=5, frameon=False, fontsize=6.5, bbox_to_anchor=(0.54, 0.985))
fig.legend(handles=plotstyle.limiter_legend_handles(), loc="lower center", ncol=4, frameon=False, fontsize=6.5,
           title="Active limiter", title_fontsize=6.5, bbox_to_anchor=(0.54, 0.0))
fig.savefig(os.path.join(FIG_DIR, "Fig_flexibility_evolution.pdf"), format="pdf", bbox_inches="tight")
plt.show()

## Figure 5 -- energy performance of the strategies

In [ ]:
FIG_WIDTH, FIG_HEIGHT = plotstyle.IEEE_COL_WIDTH_IN, 6.0
fig, axs = plt.subplots(4, 1, figsize=(FIG_WIDTH, FIG_HEIGHT), sharex=True, gridspec_kw={"hspace": 0.12})
t = logs["B0"].t_h.values
for name in ("B1", "B2", "DFM"):
    c = plotstyle.STRATEGY_COLORS[name]
    axs[0].step(t, logs[name].P_B, where="post", color=c, lw=1.0, label=plotstyle.STRATEGY_LABELS[name])
    axs[1].plot(np.r_[t, 24], np.r_[logs[name].soc, logs[name].soc_next.values[-1]], color=c, lw=1.0)
    axs[2].step(t, logs[name].P_MV0 - logs["B0"].P_MV0, where="post", color=c, lw=1.0)
for name in ("B0", "B1", "B2", "DFM"):
    axs[3].step(t, logs[name].P_curt, where="post", color=plotstyle.STRATEGY_COLORS[name], lw=1.0,
                label=plotstyle.STRATEGY_LABELS[name])
axs[0].set_ylabel(r"$P_{B,k}$ [kW]"); axs[0].legend(frameon=False, fontsize=6.0, ncol=3, loc="upper left", bbox_to_anchor=(0, 1.22))
axs[1].set_ylabel("SOC"); axs[1].set_ylim(0, 1)
axs[2].set_ylabel(r"$\Delta P_{grid}$ vs B0 [kW]"); axs[2].axhline(0, color=plotstyle.INK_MUTED, lw=0.6)
axs[3].set_ylabel(r"$P_{curt,k}$ [kW]"); axs[3].legend(frameon=False, fontsize=6.0, ncol=1, loc="upper right")
axs[3].set_xlim(0, 24); axs[3].set_xticks(range(0, 25, 4)); axs[3].set_xlabel("Time [h]")
for a, lab in zip(axs, "abcd"):
    a.text(-0.30, 1.02, f"({lab})", transform=a.transAxes, fontsize=8, va="top")
plotstyle.savefig(fig, os.path.join(FIG_DIR, "Fig_energy_performance.pdf"))
plt.show()

## Figure 6 -- AC/DC network-security indicators

In [ ]:
FIG_WIDTH, FIG_HEIGHT = plotstyle.IEEE_COL_WIDTH_IN, 6.0
fig, axs = plt.subplots(4, 1, figsize=(FIG_WIDTH, FIG_HEIGHT), sharex=True, gridspec_kw={"hspace": 0.12})
n = S.sp.net
for name in ("B0", "B1", "B2", "DFM"):
    L_, c = logs[name], plotstyle.STRATEGY_COLORS[name]
    lab = plotstyle.STRATEGY_LABELS[name]
    axs[0].step(t, L_.V_min / n.V_nom_ln, where="post", color=c, lw=0.9, label=lab)
    axs[0].step(t, L_.V_max / n.V_nom_ln, where="post", color=c, lw=0.9)
    axs[1].step(t, L_.Vdc_max / n.V_dc_nom, where="post", color=c, lw=0.9)
    load = np.maximum.reduce([L_.I_R01 / n.I_feeder_max["R01"], L_.I_I01 / n.I_feeder_max["I01"], L_.I_C01 / n.I_feeder_max["C01"]])
    axs[2].step(t, load, where="post", color=c, lw=0.9)
    axs[3].step(t, L_.util_vsc_max, where="post", color=c, lw=0.9)
axs[0].axhline(n.V_min_pu, color=plotstyle.LIMIT_COLOR, lw=0.8, ls="--"); axs[0].axhline(n.V_max_pu, color=plotstyle.LIMIT_COLOR, lw=0.8, ls="--")
axs[1].axhline(n.V_dc_max_pu, color=plotstyle.LIMIT_COLOR, lw=0.8, ls="--")
axs[2].axhline(1.0, color=plotstyle.LIMIT_COLOR, lw=0.8, ls="--"); axs[3].axhline(1.0, color=plotstyle.LIMIT_COLOR, lw=0.8, ls="--")
axs[0].set_ylabel("AC voltage\nmin/max [pu]"); axs[1].set_ylabel("max DC volt. [pu]")
axs[2].set_ylabel("max feeder\nloading [pu]"); axs[3].set_ylabel("max VSC\nutilization [pu]")
axs[0].legend(frameon=False, fontsize=6.0, ncol=2, loc="upper left", bbox_to_anchor=(0, 1.32))
axs[3].set_xlim(0, 24); axs[3].set_xticks(range(0, 25, 4)); axs[3].set_xlabel("Time [h]")
for a, lab in zip(axs, "abcd"):
    a.text(-0.30, 1.02, f"({lab})", transform=a.transAxes, fontsize=8, va="top")
plotstyle.savefig(fig, os.path.join(FIG_DIR, "Fig_network_security.pdf"))
plt.show()

## Takeaway

See the printed summary and the energy-performance, network-security and flexibility-envelope
figures; the interpretation of these results is given in the paper.